<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/08_text2sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Text2SQL (自然言語からSQL生成、売上トランザクション)

このノートブックは **pytidb シリーズの第 8 回** です。LLMを利用して、自然言語の質問をSQLに変換し、TiDB で実行して結果を返すミニ text2sql を組み立てます。

## 学習目標
- テーブル定義 (`SHOW CREATE TABLE`) をプロンプトに埋め込む
- `google.colab.ai.generate_text` で SQL を生成する
- `db.query(sql)` で生成された SQL を実行する
- 生成された SQL の危険性 (書き込み系など) をチェックする
- 結果を LLM でマークダウン整形する

前提: `00_quickstart.ipynb` / `02_query_and_filter.ipynb` を実行済み。
本ノートブックはLLMとしてColab標準のGeminiを利用します。そのため **Google Colab 前提** です。


## 1. 依存パッケージのインストールとTiDB Cloudクラスタの作成


In [ ]:
!pip install -q pytidb


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-text2sql")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 2. 売上トランザクションのサンプルデータを用意する

売上 (sales)、商品 (products)、店舗 (stores) の 3 テーブル構成にします。それぞれは、特殊なフィールドを持ちません。


In [ ]:
from datetime import date
from pytidb import TiDBClient
from pytidb.schema import Field, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="sales_demo",
    ensure_db=True,
)


class Store(TableModel):
    __tablename__ = "stores"
    __table_args__ = {"extend_existing": True}
    id: int = Field(primary_key=True)
    name: str = Field()
    region: str = Field()


class Product(TableModel):
    __tablename__ = "products"
    __table_args__ = {"extend_existing": True}
    id: int = Field(primary_key=True)
    name: str = Field()
    category: str = Field()
    unit_price: int = Field()


class Sale(TableModel):
    __tablename__ = "sales"
    __table_args__ = {"extend_existing": True}
    id: int = Field(primary_key=True)
    store_id: int = Field()
    product_id: int = Field()
    sold_on: date = Field()
    quantity: int = Field()


stores_tbl   = db.create_table(schema=Store,   if_exists="overwrite")
products_tbl = db.create_table(schema=Product, if_exists="overwrite")
sales_tbl    = db.create_table(schema=Sale,    if_exists="overwrite")
print("3 テーブル準備OK")


In [ ]:
stores_tbl.bulk_insert([
    Store(id=1, name="新宿店", region="関東"),
    Store(id=2, name="梅田店", region="関西"),
    Store(id=3, name="博多店", region="九州"),
])
products_tbl.bulk_insert([
    Product(id=1, name="ワイヤレスイヤホン", category="electronics", unit_price=12800),
    Product(id=2, name="圧力調理マルチポット", category="kitchen",   unit_price=4980),
    Product(id=3, name="登山テント 2 人用",    category="outdoor",   unit_price=15800),
    Product(id=4, name="コットンTシャツ",      category="fashion",   unit_price=1280),
])
sales_tbl.bulk_insert([
    Sale(id=1, store_id=1, product_id=1, sold_on=date(2026, 3, 1),  quantity=3),
    Sale(id=2, store_id=1, product_id=2, sold_on=date(2026, 3, 2),  quantity=2),
    Sale(id=3, store_id=2, product_id=1, sold_on=date(2026, 3, 5),  quantity=4),
    Sale(id=4, store_id=2, product_id=3, sold_on=date(2026, 3, 8),  quantity=1),
    Sale(id=5, store_id=3, product_id=4, sold_on=date(2026, 3, 10), quantity=10),
    Sale(id=6, store_id=1, product_id=4, sold_on=date(2026, 3, 15), quantity=6),
    Sale(id=7, store_id=3, product_id=2, sold_on=date(2026, 3, 20), quantity=3),
    Sale(id=8, store_id=2, product_id=4, sold_on=date(2026, 3, 25), quantity=5),
    Sale(id=9, store_id=1, product_id=3, sold_on=date(2026, 4, 1),  quantity=2),
    Sale(id=10, store_id=3, product_id=1, sold_on=date(2026, 4, 3), quantity=2),
])
print(f"投入完了: stores={stores_tbl.rows()} / products={products_tbl.rows()} / sales={sales_tbl.rows()}")


## 3. スキーマをプロンプトに埋め込むヘルパー関数を作成する

LLMが正しいスキーマ定義に基づいたSQLを書けるように、各テーブルの `CREATE TABLE` 文をLLMにそのまま渡します。そのため、`SHOW CREATE TABLE` の結果を整形してプロンプトに埋め込むヘルパー関数を作ります。


In [ ]:
def get_schema_snippet() -> str:
    parts = []
    for t in db.list_tables():
        row = db.query(f"SHOW CREATE TABLE `{t}`").to_rows()[0]
        # 行は (table_name, create_sql)
        parts.append(row[1])
    return "\n\n".join(parts)

print(get_schema_snippet()[:1200], "...")


## 4. 質問を SQL に変換する

プロンプトにルール (SELECT のみ、未定義列を作らない等) を明記します。
生成結果は SQL テキストだけを期待します。


In [ ]:
from google.colab import ai
import re

SQL_PROMPT = """あなたは TiDB / MySQL の熟練 DBA です。与えられたスキーマだけを使って、ユーザーの質問に答える **SELECT 文** を 1 つだけ出力してください。

ルール:
- SELECT 文以外 (INSERT / UPDATE / DELETE / DROP など) は絶対に使わない
- 未定義のテーブルやカラムは作らない
- バッククォートでテーブル・カラム名を囲む
- SQL 以外の説明文、記号、コードフェンスは一切つけず、SQL そのものだけを返す

--- スキーマ ---
{schema}
--- ここまで ---

質問: {question}
SQL:
"""


def nl_to_sql(question: str) -> str:
    raw = ai.generate_text(SQL_PROMPT.format(schema=get_schema_snippet(), question=question))
    # コードフェンスが混じっていたら剥がす
    m = re.search(r"```(?:sql)?(.*?)```", raw, re.S)
    sql = (m.group(1) if m else raw).strip().rstrip(";")
    return sql


question = "関東地方の3月の売上金額の合計は?"
sql = nl_to_sql(question)
print("生成された SQL:\n", sql)


## 5. 安全チェックして実行する

生成された SQL が本当に読み取り専用(`select`)か軽く検査してから、`db.query` で実行します。


In [ ]:
WRITE_KEYWORDS = ("insert ", "update ", "delete ", "drop ", "truncate ", "alter ")


def is_read_only(sql: str) -> bool:
    lower = sql.lower()
    if not lower.lstrip().startswith("select"):
        return False
    return not any(k in lower for k in WRITE_KEYWORDS)


def run_nl_query(question: str):
    sql = nl_to_sql(question)
    print("SQL:", sql)
    if not is_read_only(sql):
        print("[!] 読み取り専用ではない SQL が生成されました。実行を中止します。")
        return
    rows = db.query(sql).to_rows()
    print(f"結果 ({len(rows)} 行):")
    for r in rows:
        print(" ", r)


run_nl_query("関東地方の3月の売上金額の合計は?")
print()
run_nl_query("カテゴリ別に売上金額を集計して金額の大きい順に並べて")
print()
run_nl_query("店舗ごとに最もよく売れた商品を1つずつ教えて")


## チャレンジ

- 質問を変えてみる: 「日別の売上トレンド」「商品別の販売数量ランキング」
- SQL 生成に失敗するケースを探し、`SQL_PROMPT` にルールを追加して改善する
- 結果を LLM で表形式のマークダウンに整形するセルを追加する
